# RAG Q&A — Contoso Troubleshooting Runbook

Stack: `pypdf` → `all-MiniLM-L6-v2` embeddings → FAISS → `gpt-4o-mini`. Needs `OPENAI_API_KEY`. Streamlit version in `app.py`.

In [ ]:
# pip install -r requirements.txt   (pypdf, sentence-transformers, faiss-cpu, openai, streamlit)

import os, re, textwrap
import numpy as np
import pandas as pd
from pypdf import PdfReader

PDF_PATH = "troubleshooting_runbook.pdf"

## Part 1 — Data understanding and preprocessing

In [ ]:
def load_pages(path):
    reader = PdfReader(path)
    return [page.extract_text() or "" for page in reader.pages]

raw_pages = load_pages(PDF_PATH)

print("Pages:", len(raw_pages))
pd.DataFrame({
    "page": range(1, len(raw_pages) + 1),
    "characters": [len(p) for p in raw_pages],
    "words": [len(p.split()) for p in raw_pages],
})

In [ ]:
print(raw_pages[2][:900])

Text quality: repeated header/footer on every page, tables flattened to one row per line, code blocks hard-wrapped. No OCR issues. Cleaning only removes the header and extra whitespace.

In [ ]:
HEADER = re.compile(
    r"Contoso Platform Engineering \| Troubleshooting Runbook v3\.2 \| Internal - Synthetic demo data\s*\nPage \d+\n?"
)

def clean(text):
    text = HEADER.sub("", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

pages = [clean(p) for p in raw_pages]
full_text = "\n\n".join(pages)
print("Total words after cleaning:", len(full_text.split()))

### Chunking

Fixed-size windows cut through error-code entries. Section-aware splitting (one chunk per heading, sub-split with overlap if too long) is used instead. The document is ~2,300 words, so 500–1000 token chunks would give only 3–5 chunks.

In [ ]:
def approx_tokens(text):
    # ~1.3 tokens per word
    return int(len(text.split()) * 1.3)

def sliding_window(text, chunk_words, overlap_words):
    words = text.split()
    step = chunk_words - overlap_words
    out = []
    for start in range(0, len(words), step):
        out.append(" ".join(words[start:start + chunk_words]))
        if start + chunk_words >= len(words):
            break
    return out

rows = []
for cw, ow in [(150, 30), (250, 50), (400, 80), (600, 100), (800, 150)]:
    chunks = sliding_window(full_text, cw, ow)
    rows.append({"chunk_words": cw, "overlap_words": ow,
                 "approx_tokens": approx_tokens(" ".join(["w"] * cw)), "n_chunks": len(chunks)})
pd.DataFrame(rows)

In [ ]:
HEADING = re.compile(
    r"\n(?=(?:ERR-\d{3}|DB-\d{4}|DB-POOL|MEM-OOM|DISK-FULL|CERT-EXP) - "
    r"|\d\. [A-Z][^\n.]{3,60}\n"
    r"|[A-D]\. [A-Z][^\n]+\n)"
)

def chunk_pages(pages, max_words=300, overlap_words=50):
    chunks = []
    for page_no, page in enumerate(pages, start=1):
        sections = [s.strip() for s in HEADING.split("\n" + page) if s.strip()]
        # short category labels get attached to the next section
        label = ""
        for sec in sections:
            if len(sec.split()) < 15:
                label = sec.replace("\n", " ")
                continue
            title = sec.split("\n", 1)[0].strip()
            body = f"{label}\n{sec}" if label else sec
            pieces = sliding_window(body, max_words, overlap_words) if len(body.split()) > max_words else [body]
            for i, piece in enumerate(pieces):
                chunks.append({
                    "id": len(chunks),
                    "page": page_no,
                    "section": title,
                    "part": i,
                    "text": piece,
                    "tokens": approx_tokens(piece),
                })
    return chunks

chunks = chunk_pages(pages)
df_chunks = pd.DataFrame(chunks)
print(len(chunks), "chunks")
df_chunks[["id", "page", "section", "tokens"]]

In [ ]:
print(df_chunks["tokens"].describe())
print()
print(chunks[15]["section"])
print("-" * 60)
print(chunks[15]["text"])

## Part 2 — Embeddings and vector database

Embeddings: `all-MiniLM-L6-v2` (384-d, local, trained for semantic similarity). Store: FAISS `IndexFlatIP` on normalised vectors = cosine similarity; exact search is fine at this size. Chunk metadata is kept in a list indexed by FAISS row id.

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss

embedder = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = embedder.encode([c["text"] for c in chunks], normalize_embeddings=True, show_progress_bar=True)
embeddings = np.asarray(embeddings, dtype="float32")
print(embeddings.shape)

index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)
print("vectors in index:", index.ntotal)

In [ ]:
def embed_query(query):
    return embedder.encode([query], normalize_embeddings=True).astype("float32")

def search(query, k=5):
    scores, ids = index.search(embed_query(query), k)
    return [{"score": float(s), **chunks[i]} for s, i in zip(scores[0], ids[0]) if i != -1]

def show_results(results, width=110):
    for r in results:
        print(f"[{r['score']:.3f}] p.{r['page']}  {r['section']}")
        print(textwrap.shorten(r["text"].replace("\n", " "), width=width))
        print()

show_results(search("pods keep restarting with exit code 137", k=3))

## Part 3 — Retrieval and choice of k

In [ ]:
test_queries = [
    "What should I do when the connection pool is exhausted?",
    "Who gets paged for a Sev1 incident and how often are updates posted?",
    "Why would payment-gateway return 504 errors?",
]

for q in test_queries:
    print("=" * 100)
    print("Q:", q)
    for k in (3, 5, 7):
        res = search(q, k)
        print(f"\n  k={k}:")
        for r in res:
            print(f"    [{r['score']:.3f}] p.{r['page']} {r['section']}")

k=3 already contains the answering section; k=5 adds related entries (quick reference, neighbouring codes); k=7 pulls in chunks that only share vocabulary. Default: k=5 with a 0.25 similarity floor.

In [ ]:
TOP_K = 5
MIN_SCORE = 0.25

def retrieve(query, k=TOP_K, min_score=MIN_SCORE):
    return [r for r in search(query, k) if r["score"] >= min_score]

## Part 4 — Answer generation

Prompt: system message restricts answers to the supplied excerpts, fixed refusal phrase when the answer is absent, each excerpt numbered `[n]` with page and section, every factual sentence must cite its `[n]`, exact values preserved, temperature 0.

In [ ]:
from openai import OpenAI

client = OpenAI()  # uses OPENAI_API_KEY from the environment
LLM_MODEL = "gpt-4o-mini"

SYSTEM_PROMPT = """You are an on-call assistant that answers questions using the Contoso Platform Troubleshooting Runbook.

Rules:
- Answer ONLY from the runbook excerpts provided in the user message. Do not use outside knowledge.
- If the excerpts do not contain the answer, reply exactly: "The runbook does not cover this." and nothing else.
- After every sentence that states a fact, add the excerpt number it came from in square brackets, e.g. [2].
- Keep numbers, thresholds, commands and config names exactly as they appear in the excerpts.
- Be concise. Use a short list for step-by-step procedures, otherwise plain sentences."""

def build_context(results):
    blocks = []
    for n, r in enumerate(results, start=1):
        blocks.append(f"[{n}] (page {r['page']}, section: {r['section']})\n{r['text']}")
    return "\n\n".join(blocks)

def generate_answer(question, results):
    if not results:
        return "The runbook does not cover this."
    user_msg = f"Runbook excerpts:\n\n{build_context(results)}\n\nQuestion: {question}"
    resp = client.chat.completions.create(
        model=LLM_MODEL,
        temperature=0,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_msg},
        ],
    )
    return resp.choices[0].message.content.strip()

In [ ]:
def ask(question, k=TOP_K, show_sources=True):
    results = retrieve(question, k)
    answer = generate_answer(question, results)
    print("Q:", question)
    print("\nA:", answer)
    if show_sources and results:
        print("\nSources:")
        for n, r in enumerate(results, start=1):
            print(f"  [{n}] p.{r['page']} {r['section']}  (score {r['score']:.3f})")
    print("\n" + "-" * 100 + "\n")
    return answer, results

sample_questions = [
    "What are the likely causes of ERR-401 and how do I fix it?",
    "How long do we have to acknowledge a Sev2 and who is escalated to?",
    "What is the rule for sizing the database connection pool?",
    "A pod was OOMKilled. What should I check before restarting it?",
    "Which team owns search-service and what does it depend on?",
    "What does the rollback note say about database migrations?",
]

for q in sample_questions:
    ask(q)

### Grounding check

Out-of-scope questions should return the refusal phrase; cited excerpts should contain the stated facts.

In [ ]:
# out-of-scope
ask("How do I configure Kafka consumer groups?")
ask("What is the monthly cost of the payments-db instance?")

In [ ]:
# citation spot-check
answer, results = ask("What are the quarterly targets for MTTR and deploy-related incidents?", show_sources=False)

cited = sorted({int(n) for n in re.findall(r"\[(\d+)\]", answer)})
print("Cited excerpts:", cited)
for n in cited:
    r = results[n - 1]
    print(f"\n[{n}] p.{r['page']} {r['section']}")
    print(r["text"])

## Part 5 — End-to-end application

`ask()` is the full pipeline. Interactive loop below; `streamlit run app.py` for the UI.

In [ ]:
def chat():
    print("Ask the runbook. Type 'exit' to stop.\n")
    while True:
        q = input("> ").strip()
        if not q or q.lower() in {"exit", "quit"}:
            break
        ask(q)

chat()